# 05 — Compare Triton with CUDA C++

**Goal:** implement the same vector-add operation using a small CUDA C++ extension and compare it with the Triton version.

This notebook is optional because building a CUDA extension requires a compatible CUDA compiler (`nvcc`) and matching PyTorch environment. If the compiler is missing, the notebook explains what to check and the other experiments remain usable.

## 1. Check the runtime and compiler

In [ ]:
%pip install ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 8.2 MB/s eta 0:00:00


In [ ]:
import shutil
import torch
import triton
import triton.language as tl
import ninja
print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("CUDA available to PyTorch:", torch.cuda.is_available())
print("nvcc path:", shutil.which("nvcc"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Ninja version:", ninja.__version__)
print("Ninja executable:", shutil.which("ninja"))

PyTorch: 2.11.0+cu130
Triton: 3.6.0
CUDA available to PyTorch: True
nvcc path: /usr/local/cuda/bin/nvcc
GPU: Tesla T4
Ninja version: 1.13.2
Ninja executable: /usr/local/bin/ninja


## 2. Define a small CUDA extension

CUDA C++ explicitly writes a kernel and declares how many threads are in each CUDA block. Here, each thread calculates one array element. The C++ wrapper checks the inputs and launches the GPU kernel on PyTorch's current CUDA stream.

The code is kept here rather than hidden in a downloaded binary so you can read it, modify it, and understand the comparison.

In [ ]:
import os

# Explicitly configure the CUDA toolkit and target the Tesla T4.
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

cpp_source = r"""
#include <torch/extension.h>
#include <c10/cuda/CUDAException.h>

void vector_add_cuda(torch::Tensor x, torch::Tensor y, torch::Tensor out);

void vector_add_(torch::Tensor x, torch::Tensor y, torch::Tensor out) {
    TORCH_CHECK(x.is_cuda() && y.is_cuda() && out.is_cuda(), "Tensors must be on CUDA");
    TORCH_CHECK(x.scalar_type() == torch::kFloat32, "x must be float32");
    TORCH_CHECK(y.scalar_type() == torch::kFloat32, "y must be float32");
    TORCH_CHECK(out.scalar_type() == torch::kFloat32, "out must be float32");
    TORCH_CHECK(x.is_contiguous() && y.is_contiguous() && out.is_contiguous(),
                "Tensors must be contiguous");
    TORCH_CHECK(x.sizes() == y.sizes() && x.sizes() == out.sizes(),
                "Tensor shapes must match");
    vector_add_cuda(x, y, out);
}
"""

cuda_source = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAException.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void vector_add_kernel(const float* x, const float* y,
                                 float* out, long long n) {
    long long i = static_cast<long long>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (i < n) {
        out[i] = x[i] + y[i];
    }
}

void vector_add_cuda(torch::Tensor x, torch::Tensor y, torch::Tensor out) {
    long long n = x.numel();
    if (n == 0) return;
    int threads = 256;
    int blocks = static_cast<int>((n + threads - 1) / threads);
    vector_add_kernel<<<blocks, threads, 0, at::cuda::getCurrentCUDAStream()>>>(
        x.data_ptr<float>(), y.data_ptr<float>(), out.data_ptr<float>(), n);
    C10_CUDA_KERNEL_LAUNCH_CHECK();
}
"""

print("C++ and CUDA source definitions ready.")

C++ and CUDA source definitions ready.


## 3. Build the extension when supported

Compilation may take a few minutes the first time. Do not install a random CUDA toolkit into a working Colab runtime if the versions are incompatible. If compilation fails, save the error message and investigate it rather than silently changing the environment.

In [ ]:
import os
from pathlib import Path
import torch

build_root = Path("/root/.cache/torch_extensions/py313_cu130")

matches = list(build_root.glob("triton_lab_cuda_add_debug_v3/*.so"))
print("Built libraries:", matches)

if not matches:
    raise FileNotFoundError(
        "The v3 shared library was not found. Rerun the successful v3 build cell."
    )

import importlib.util

spec = importlib.util.spec_from_file_location(
    "triton_lab_cuda_add_debug_v3", str(matches[0])
)
cuda_add_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cuda_add_module)

print("Loaded CUDA C++ extension:", matches[0])

Built libraries: [PosixPath('/root/.cache/torch_extensions/py313_cu130/triton_lab_cuda_add_debug_v3/triton_lab_cuda_add_debug_v3.so')]
Loaded CUDA C++ extension: /root/.cache/torch_extensions/py313_cu130/triton_lab_cuda_add_debug_v3/triton_lab_cuda_add_debug_v3.so


## 4. Add a Triton reference in the same notebook

This keeps the operation constant while changing the way the GPU kernel is expressed.

In [ ]:
@triton.jit
def vector_add_kernel(x_ptr, y_ptr, out_ptr, n_elements: tl.constexpr,
                      BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.load(y_ptr + offsets, mask=mask, other=0.0)
    tl.store(out_ptr + offsets, x + y, mask=mask)


def launch_triton_add(x, y, out, block_size=256):
    grid = (triton.cdiv(x.numel(), block_size),)
    vector_add_kernel[grid](x, y, out, x.numel(), BLOCK_SIZE=block_size)

## 5. Compare correctness and timing

In [ ]:
import statistics

def median_gpu_time_ms(fn, warmup=10, repeats=50, rounds=5):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(rounds):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats):
            fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) / repeats)
    return statistics.median(samples), samples

if cuda_add_module is not None:
    n = 1 << 20
    x = torch.randn(n, device="cuda", dtype=torch.float32)
    y = torch.randn_like(x)
    out_torch = torch.empty_like(x)
    out_triton = torch.empty_like(x)
    out_cuda = torch.empty_like(x)

    def run_torch():
        torch.add(x, y, out=out_torch)

    def run_triton():
        launch_triton_add(x, y, out_triton, block_size=256)

    def run_cuda_cpp():
        cuda_add_module.vector_add_(x, y, out_cuda)

    run_torch(); run_triton(); run_cuda_cpp()
    torch.testing.assert_close(out_triton, out_torch)
    torch.testing.assert_close(out_cuda, out_torch)

    for label, fn in [("PyTorch", run_torch), ("Triton", run_triton), ("CUDA C++", run_cuda_cpp)]:
        ms, rounds = median_gpu_time_ms(fn)
        print(f"{label:10s}: {ms:.6f} ms/call; rounds={['%.6f' % v for v in rounds]}")
else:
    print("Timing skipped because the CUDA extension was not built.")

PyTorch   : 0.054950 ms/call; rounds=['0.054950', '0.055010', '0.054934', '0.055073', '0.054722']
Triton    : 0.052101 ms/call; rounds=['0.051910', '0.052351', '0.052211', '0.052101', '0.051949']
CUDA C++  : 0.051766 ms/call; rounds=['0.051773', '0.051896', '0.051759', '0.051751', '0.051766']


## Final notes

Triton often makes custom GPU kernels easier to write. CUDA C++ gives more direct control but requires more explicit kernel and build details. A single vector-add timing result does not decide which approach is better overall. Record compiler/toolkit versions, correctness, build issues, and measured timing. The next research projects—FlexKV and PagedKV—should begin only after these kernel basics are understood.